# LM-Meter: Quick-Start Post-Processing Notebook

This notebook demonstrates how to post-process and analyze latency data collected by LM-Meter.

## What this notebook covers
1. **Phase-level analysis** – parse `tvm_mlc.log` and extract per-phase latencies
   (Embedding, Prefill, Decode, Softmax, CopyProbsToCPU, Sampling)
2. **Kernel-level analysis** – parse `trace_*.json` (Chrome/Perfetto format) and extract
   per-kernel GPU execution times
3. **Accuracy metrics** – compute α (%) and ε★ (μs/ms) as defined in the paper
4. **Visualisation** – bar charts, tables and comparisons

### References
- Paper: *lm-Meter: Unveiling Runtime Inference Latency for On-Device Language Models* (SEC 2025)
- Data collection guide: [docs/data-collection.md](../docs/data-collection.md)
- Evaluation guide:      [docs/eval.md](../docs/eval.md)


In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import json
import os
import re
import glob
from pathlib import Path
from collections import defaultdict

# ── Scientific stack ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## ① Configuration

Set the paths to your experiment output directory produced by
`option2_stream_logcats_and_pull_traces.sh`.

In [ ]:
# ── Update these paths to point to your actual experiment folder ───────────
EXPERIMENT_DIR = Path("../output/option2_20250101_120000")  # <-- change me
LOG_FILE       = EXPERIMENT_DIR / "tvm_mlc.log"
TRACES_DIR     = EXPERIMENT_DIR / "traces"

# Ground-truth latencies (ms) from the paper – used to compute accuracy.
# Replace with your own ground-truth measurements if available.
GROUND_TRUTH_PHASE = {
    "Llama-3.2-3B-Instruct": {
        "Embedding":      0.7763,
        "Prefill":     3433.8142,
        "Decode":        62.5303,
        "Softmax":      142.6542,
        "CopyProbsToCPU": 0.4616,
        "Sampling":       0.0824,
    },
    "Gemma-2-2B-it": {
        "Embedding":      0.7398,
        "Prefill":     9301.0589,
        "Decode":        54.5557,
        "Softmax":      502.3698,
        "CopyProbsToCPU": 0.5255,
        "Sampling":       0.1830,
    },
}

print(f"Experiment directory : {EXPERIMENT_DIR.resolve()}")
print(f"Log file exists      : {LOG_FILE.exists()}")
print(f"Traces directory     : {TRACES_DIR.resolve()}")
print(f"Trace files found    : {len(list(TRACES_DIR.glob('trace_*.json')))}")

---
## ② Helper functions: accuracy metrics

The paper defines two accuracy metrics:

| Metric | Formula | Unit |
|--------|---------|------|
| **α** | `100 × (1 − |T_profiled − T_gt| / T_gt)` | % |
| **ε★** | `|T_profiled − T_gt| / T_gt × 1000` | μs/ms |

In [ ]:
def alpha(t_profiled: float, t_gt: float) -> float:
    """Accuracy α (%) – higher is better."""
    if t_gt == 0:
        return float("nan")
    return 100.0 * (1.0 - abs(t_profiled - t_gt) / t_gt)


def epsilon_star(t_profiled: float, t_gt: float) -> float:
    """Normalised error ε★ (μs/ms) – lower is better."""
    if t_gt == 0:
        return float("nan")
    return abs(t_profiled - t_gt) / t_gt * 1000.0


def metrics_row(name: str, t_profiled: float, t_gt: float) -> dict:
    return {
        "Phase / Kernel": name,
        "LM-Meter (ms)": round(t_profiled, 4),
        "GT (ms)": round(t_gt, 4),
        "α (%)": round(alpha(t_profiled, t_gt), 2),
        "ε★ (μs/ms)": round(epsilon_star(t_profiled, t_gt), 3),
    }

---
## ③ Phase-level analysis – parse `tvm_mlc.log`

The logcat file contains lines tagged `MLC_Profile` with per-phase latency measurements.
Example line format:
```
01-01 12:00:00.000  1234  1234 I MLC_Profile: phase=Prefill latency_ms=3433.86
```

In [ ]:
# ── Patterns to match MLC_Profile log lines ─────────────────────────────
# Support multiple common formats emitted by MLC LLM / TVM runtime
_PHASE_PATTERNS = [
    # Format A: phase=Prefill latency_ms=3433.86
    re.compile(r"MLC_Profile.*?phase=(\S+)\s+latency_ms=([\d.]+)"),
    # Format B: [MLC_Profile] Prefill: 3433.86 ms
    re.compile(r"\[MLC_Profile\]\s+(\S+):\s*([\d.]+)\s*ms"),
    # Format C: TVM_RUNTIME phase Prefill 3433.86
    re.compile(r"TVM_RUNTIME.*?phase\s+(\S+)\s+([\d.]+)"),
    # Format D: MLC_EVENT phase_end Prefill elapsed=3433.86
    re.compile(r"MLC_EVENT.*?phase_end\s+(\S+)\s+elapsed=([\d.]+)"),
]


def parse_phase_log(log_path: Path) -> dict[str, list[float]]:
    """Parse tvm_mlc.log and return {phase_name: [latency_ms, ...]}."""
    results: dict[str, list[float]] = defaultdict(list)

    if not log_path.exists():
        print(f"[WARN] Log file not found: {log_path}")
        return results

    with open(log_path, "r", errors="replace") as fh:
        for line in fh:
            for pat in _PHASE_PATTERNS:
                m = pat.search(line)
                if m:
                    phase, latency = m.group(1), float(m.group(2))
                    results[phase].append(latency)
                    break  # only one pattern per line
    return dict(results)


phase_data = parse_phase_log(LOG_FILE)
print("Phases detected:", list(phase_data.keys()) if phase_data else "(none – check log path)")

In [ ]:
# ── Aggregate: take the mean over all captured tokens / iterations ────────
phase_mean: dict[str, float] = {k: float(np.mean(v)) for k, v in phase_data.items()}

if phase_mean:
    df_phase = pd.DataFrame([
        {"Phase": phase, "Mean Latency (ms)": round(lat, 4)}
        for phase, lat in phase_mean.items()
    ])
    display(df_phase)
else:
    print("No phase data – using built-in demo values for illustration.")
    # Demo data (Llama-3.2-3B-Instruct from the paper)
    phase_mean = {
        "Embedding":       0.8038,
        "Prefill":      3433.8628,
        "Decode":          62.5669,
        "Softmax":        142.6166,
        "CopyProbsToCPU":   0.4929,
        "Sampling":         0.0675,
    }

In [ ]:
# ── Compute accuracy metrics against ground truth ─────────────────────────
MODEL_NAME = "Llama-3.2-3B-Instruct"   # <-- change to match your model
gt_phase = GROUND_TRUTH_PHASE.get(MODEL_NAME, {})

rows = []
for phase, t_prof in phase_mean.items():
    t_gt = gt_phase.get(phase, float("nan"))
    if not np.isnan(t_gt):
        rows.append(metrics_row(phase, t_prof, t_gt))

if rows:
    df_metrics = pd.DataFrame(rows)
    print(f"\nPhase-level accuracy metrics ({MODEL_NAME}):")
    display(df_metrics.style.format({
        "LM-Meter (ms)": "{:.4f}",
        "GT (ms)":       "{:.4f}",
        "α (%)":         "{:.2f}",
        "ε★ (μs/ms)":    "{:.3f}",
    }))
else:
    print("Skipping accuracy computation (no ground-truth values configured).")

In [ ]:
# ── Bar chart: profiled vs ground-truth latencies ─────────────────────────
phases_to_plot = [p for p in phase_mean if p in gt_phase]
if phases_to_plot:
    x = np.arange(len(phases_to_plot))
    width = 0.35
    profiled_vals = [phase_mean[p] for p in phases_to_plot]
    gt_vals       = [gt_phase[p]   for p in phases_to_plot]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width / 2, profiled_vals, width, label="LM-Meter", color="steelblue")
    ax.bar(x + width / 2, gt_vals,       width, label="Ground Truth", color="coral", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(phases_to_plot, rotation=20, ha="right")
    ax.set_ylabel("Latency (ms)")
    ax.set_title(f"Phase-level latency: {MODEL_NAME}")
    ax.legend()
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:,.1f}"))
    plt.tight_layout()
    plt.savefig(EXPERIMENT_DIR / "phase_latency.png", dpi=150)
    plt.show()
    print("Plot saved to", EXPERIMENT_DIR / "phase_latency.png")

---
## ④ Kernel-level analysis – parse `trace_*.json`

Each trace file contains a list of Chrome/Perfetto trace events.
We extract events with `ph == 'X'` (complete events) that carry a `dur` field (duration in μs).

In [ ]:
def parse_trace_files(traces_dir: Path) -> pd.DataFrame:
    """
    Parse all trace_*.json files and return a DataFrame with columns:
      name, ph, ts, dur_us, dur_ms, pid, tid
    """
    records = []
    trace_files = sorted(traces_dir.glob("trace_*.json"))

    if not trace_files:
        print(f"[WARN] No trace_*.json files found in {traces_dir}")
        return pd.DataFrame()

    for tf in trace_files:
        try:
            with open(tf, "r", errors="replace") as fh:
                events = json.load(fh)
        except (json.JSONDecodeError, ValueError) as exc:
            print(f"[WARN] Could not parse {tf.name}: {exc}")
            continue

        for ev in events:
            if not isinstance(ev, dict):
                continue
            if ev.get("ph") not in ("X", "B", "E"):
                continue
            dur_us = ev.get("dur", 0)
            records.append({
                "file":   tf.name,
                "name":   ev.get("name", "unknown"),
                "ph":     ev.get("ph"),
                "ts":     ev.get("ts", 0),
                "dur_us": dur_us,
                "dur_ms": dur_us / 1000.0,
                "pid":    ev.get("pid"),
                "tid":    ev.get("tid"),
            })

    if not records:
        return pd.DataFrame()

    df = pd.DataFrame(records)
    df = df[df["dur_us"] > 0].copy()  # drop zero-duration events
    return df


df_traces = parse_trace_files(TRACES_DIR)
print(f"Total kernel events parsed: {len(df_traces)}")
if not df_traces.empty:
    display(df_traces.head(10))

In [ ]:
# ── Aggregate kernel latencies ────────────────────────────────────────────
if not df_traces.empty:
    df_kernel_agg = (
        df_traces
        .groupby("name")["dur_ms"]
        .agg(count="count", mean_ms="mean", total_ms="sum", std_ms="std")
        .reset_index()
        .sort_values("total_ms", ascending=False)
        .round(4)
    )
    print("\nKernel-level aggregated latencies (top 20 by total time):")
    display(df_kernel_agg.head(20))
else:
    print("No kernel trace data available.")
    # Demo data (Gemma-2-2B-it / Pixel 8 Pro from the paper)
    df_kernel_agg = pd.DataFrame([
        {"name": "dequantize4_NT_matmul8",   "mean_ms": 367.5603},
        {"name": "dequantize3_NT_matmul7",   "mean_ms": 330.3757},
        {"name": "dequantize_NT_matmul14…",   "mean_ms":  18.4379},
        {"name": "dequantize1_NT_matmul5",   "mean_ms":  81.1899},
        {"name": "dequantize2_NT_matmul6",   "mean_ms":  31.3407},
    ])

In [ ]:
# ── Bar chart: top-N kernels by mean latency ──────────────────────────────
TOP_N = 15
col = "mean_ms" if "mean_ms" in df_kernel_agg.columns else "mean_ms"
top_kernels = df_kernel_agg.nlargest(TOP_N, col)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top_kernels["name"], top_kernels[col], color="steelblue")
ax.set_xlabel("Mean latency (ms)")
ax.set_title(f"Top-{TOP_N} kernels by mean execution time")
ax.invert_yaxis()  # highest at top
plt.tight_layout()
if not df_traces.empty:
    plt.savefig(EXPERIMENT_DIR / "kernel_latency.png", dpi=150)
plt.show()

---
## ⑤ Export results to CSV

In [ ]:
if rows:
    out_csv = EXPERIMENT_DIR / "phase_metrics.csv"
    df_metrics.to_csv(out_csv, index=False)
    print(f"Phase metrics saved to: {out_csv}")

if not df_traces.empty:
    out_csv2 = EXPERIMENT_DIR / "kernel_aggregated.csv"
    df_kernel_agg.to_csv(out_csv2, index=False)
    print(f"Kernel aggregated latencies saved to: {out_csv2}")

---
## ⑥ Summary

This notebook showed how to:
- Parse `tvm_mlc.log` for phase-level latency data
- Parse `trace_*.json` for kernel-level GPU execution data
- Compute the LM-Meter accuracy metrics **α** and **ε★**
- Visualise and export results

For further analysis, refer to:
- [docs/eval.md](../docs/eval.md) – expected results from the paper
- [docs/data-collection.md](../docs/data-collection.md) – collecting more data
- [docs/common-errors.md](../docs/common-errors.md) – troubleshooting